# MidiTokenizer demo — tokenize / decode round-trip

Tokenize `test202606/txt/机器猫.txt` with `starry.midi.tokenizer.MidiTokenizer`
(SkyTNT-style event patches), print the result, decode back, and verify against
the original text.

Note: lines whose event type is excluded (TEXT_SPECS / DATA_SPECS — track_name,
lyrics, sysex, …) are dropped by design, so the decoded text is compared against
the *retained* subset of the original, not the raw file byte-for-byte.

In [1]:
import os
import sys

# repo root = two levels up from tests/midi/
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from starry.midi.tokenizer import MidiTokenizer, theoretical_max_event_tokens

TXT_PATH = '/home/camus/data/midi/test202606/txt/机器猫.txt'
with open(TXT_PATH, encoding='utf-8') as f:
    text = f.read()

tk = MidiTokenizer()   # default patch_size (16; theoretical max retained-event len = 15)
print('vocab_size =', tk.vocab_size, '| patch_size =', tk.patch_size,
      '| theoretical max event tokens =', theoretical_max_event_tokens())
print('special ids: pad=%d bos=%d eos=%d unknown=%d' % (tk.pad_id, tk.bos_id, tk.eos_id, tk.unknown_id))
print('excluded event types:', sorted(tk.excluded))
print('source lines =', text.count(chr(10)) + 1)
print('--- first 8 source lines ---')
print(chr(10).join(text.split(chr(10))[:8]))

vocab_size = 37 | patch_size = 16 | theoretical max event tokens = 15
special ids: pad=0 bos=1 eos=2 unknown=3
excluded event types: ['copyright', 'cue_point', 'divided_sysex', 'instrument_name', 'lyrics', 'marker', 'meta_unknown', 'sequencer_specific', 'smpte_offset', 'sysex', 'text', 'time_signature', 'track_name']
source lines = 13543
--- first 8 source lines ---
ticks_per_beat 1e0
format_type 0
track_name 0 DORAEMONNO UTA
copyright 0 (C)1991 Roland Corporation
time_signature 0 4 4 18 8
set_tempo 0 790fb
sysex 0 4110421240007f0041f7
sysex 3c5 4110421240013340400cf7


## 1. Tokenize → event patches

In [2]:
unknowns = {}
patches, dropped = tk.encode_patches(text, add_special_patches=True, unknowns=unknowns)

print('total patches =', len(patches), '(includes leading <bos> + trailing <eos>)')
print('dropped (excluded TEXT/DATA) event types:',
      {d: dropped.count(d) for d in sorted(set(dropped))} or '(none)')
print('unknown content chars:', {k: v.count for k, v in unknowns.items()} or '(none)')

def patch_to_tokens(patch):
    return [tk.token_by_id[t] for t in patch]

print('\n--- patch[0] (bos) ---')
print(patches[0], '->', patch_to_tokens(patches[0]))
print('\n--- first 3 real event patches ---')
for p in patches[1:4]:
    print(p)
    print('   tokens:', patch_to_tokens(p))
print('\n--- patch[-1] (eos) ---')
print(patches[-1], '->', patch_to_tokens(patches[-1]))

total patches = 13539 (includes leading <bos> + trailing <eos>)
dropped (excluded TEXT/DATA) event types: {'copyright': 1, 'sysex': 3, 'time_signature': 1, 'track_name': 1}
unknown content chars: (none)

--- patch[0] (bos) ---
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2] -> ['<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<bos>', '<eos>']

--- first 3 real event patches ---
[16, 20, 33, 19, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
   tokens: ['ticks_per_beat', '1', 'e', '0', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
[17, 19, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
   tokens: ['format_type', '0', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
[11, 19, 35, 26, 28, 19, 34, 30, 0, 0, 0, 0, 0, 0, 0, 0]
   tokens: ['set_tempo', '0', ' ', '7', '9', '0', 'f', 'b', '<pad>',

## 2. Decode back → text

In [3]:
decoded = tk.decode_patches(patches)
print('decoded lines =', decoded.count(chr(10)) + 1)
print('--- first 8 decoded lines ---')
print(chr(10).join(decoded.split(chr(10))[:8]))

decoded lines = 13537
--- first 8 decoded lines ---
ticks_per_beat 1e0
format_type 0
set_tempo 0 790fb
control_change eb 4 0 0
control_change 0 4 20 0
program_change 0 4 39
control_change 0 3 0 0
control_change 0 3 20 0


## 3. Verify against the original (retained subset)

The decoder can only reproduce *retained* events (excluded TEXT/DATA lines are
dropped at encode time). So the correct reference is the original text with those
same lines filtered out. We assert the decoded text equals that retained subset
line-for-line.

In [4]:
# Build the reference: original lines minus the excluded event types.
retained = []
for line in text.split('\n'):
    if not line.strip():
        continue
    head = line.split(' ', 1)[0]
    if head in tk.excluded or head not in tk.id_by_token:
        continue
    retained.append(line.rstrip())

decoded_lines = [l for l in decoded.split('\n') if l.strip()]

print('retained original lines =', len(retained))
print('decoded lines           =', len(decoded_lines))

# line-by-line comparison
mismatch = None
for i, (a, b) in enumerate(zip(retained, decoded_lines)):
    if a != b:
        mismatch = (i, a, b)
        break

same_len = len(retained) == len(decoded_lines)
exact = same_len and mismatch is None
print('\nlengths match:', same_len)
if mismatch is not None:
    i, a, b = mismatch
    print('FIRST MISMATCH at line %d:' % i)
    print('  original:', repr(a))
    print('  decoded :', repr(b))
print('\n==> round-trip on retained events:', 'PASS ✓' if exact else 'FAIL ✗')
assert exact, 'decode(encode(text)) must equal the retained-event subset of the original'

retained original lines = 13537
decoded lines           = 13537

lengths match: True

==> round-trip on retained events: PASS ✓
